# 00d — Recompute WaveDetect scores on the Ghostbuster detector pool

This notebook starts from the raw Ghostbuster detector pool created by `00c_prepare_ghostbuster_data.ipynb` and runs the **public WaveDetect model** on every document. It does not load our saved WaveDetect score cache.

Public model:

```text
KaitongQin/WaveDetect
```

We pin the Hugging Face model snapshot to the upload revision used by the public inference package:

```text
d38df62
```

The public model card exposes `WaveDetectPredictor`, which tokenizes each document with truncation and returns the positive-class `ai_score`.

### Why this notebook scores one document at a time

The released predictor's `predict()` method scores a single document. We preserve that behavior intentionally. WaveDetect's wavelet/CNN input width depends on tokenized sequence length, so naively padding unrelated documents into a batch can change the effective CNN input geometry. Single-document scoring is therefore the safest exact reproduction path.

Output:

- `results/wavedetect_scores.parquet`
  - `row_id`
  - `wave_ai_score`

The notebook checkpoints frequently and can be resumed.

In [1]:
from pathlib import Path
import sys
import importlib.util
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd

DATA = ROOT / "data"
RESULTS = ROOT / "results"
EXTERNAL = DATA / "external"

RESULTS.mkdir(parents=True, exist_ok=True)
EXTERNAL.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA / "raw" / "ghostbuster_detector_pool.parquet"
OUTPUT_PATH = RESULTS / "wavedetect_scores.parquet"

HF_REPO = "KaitongQin/WaveDetect"
HF_REVISION = "d38df62"
MODEL_DIR = EXTERNAL / "wavedetect_hf"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert INPUT_PATH.exists(), (
    f"Missing {INPUT_PATH}. Run 00c_prepare_ghostbuster_data.ipynb first."
)

print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)
print("device:", DEVICE)

/home/user/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


input: ../data/raw/ghostbuster_detector_pool.parquet
output: ../results/wavedetect_scores.parquet
device: cuda


## Download the pinned public model snapshot

This downloads the authors' tokenizer, model implementation, and `wavedetect_all.bin` weights. The model is about 1 GB.

In [2]:
snapshot_path = snapshot_download(
    repo_id=HF_REPO,
    revision=HF_REVISION,
    local_dir=str(MODEL_DIR),
)

print("snapshot:", snapshot_path)

required = [
    MODEL_DIR / "wavedetect_hf.py",
    MODEL_DIR / "wavedetect_all.bin",
    MODEL_DIR / "config.json",
]

for p in required:
    assert p.exists(), f"Missing model artifact: {p}"

Fetching 13 files: 100%|███████████████████████████████████████| 13/13 [00:10<00:00,  1.23it/s]

snapshot: ../data/external/wavedetect_hf


## Import the authors' predictor directly from the pinned snapshot

In [3]:
module_path = MODEL_DIR / "wavedetect_hf.py"

spec = importlib.util.spec_from_file_location(
    "wavedetect_hf",
    module_path,
)
wave_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(wave_module)

WaveDetectPredictor = wave_module.WaveDetectPredictor

predictor = WaveDetectPredictor(
    model_dir=str(MODEL_DIR),
    device=DEVICE,
)

print("Loaded WaveDetectPredictor")

Loaded WaveDetectPredictor


## Load the exact raw documents

`row_id` is generated from public Ghostbuster path/domain/generator information, so it is independent of our \(C/F\) feature computation.

In [4]:
pool = pd.read_parquet(INPUT_PATH)

required_cols = ["row_id", "text", "source", "domain", "generator", "label"]
missing = [c for c in required_cols if c not in pool.columns]
assert not missing, missing
assert not pool["row_id"].duplicated().any()

print("pool:", pool.shape)
display(
    pool.groupby(["source", "domain", "generator"]).size().to_frame("n")
)

pool: (9000, 10)


n
source      domain generator      
claude_eval essay  claude     1000
            reuter claude     1000
            wp     claude     1000
original    essay  gpt        1000
                   human      1000
            reuter gpt        1000
                   human      1000
            wp     gpt        1000
                   human      1000

## Single-document sanity check

In [5]:
example = pool.iloc[0]
result = predictor.predict(example["text"])

print("row_id:", example["row_id"])
print("generator:", example["generator"])
print(result)

assert "ai_score" in result
assert 0.0 <= float(result["ai_score"]) <= 1.0

row_id: original::wp::gpt::1
generator: gpt
{'label': 1, 'label_name': 'ai_generated', 'ai_score': 0.9999887943267822}


## Score or resume the full corpus

Failures are recorded instead of silently dropping documents. Re-running the cell only scores rows not already present in the cache.

In [6]:
if OUTPUT_PATH.exists():
    scores = pd.read_parquet(OUTPUT_PATH)
else:
    scores = pd.DataFrame(
        columns=[
            "row_id",
            "wave_ai_score",
            "wave_error",
        ]
    )

done = set(scores["row_id"].astype(str))

todo = pool[
    ~pool["row_id"].astype(str).isin(done)
].copy()

print("already scored:", len(scores))
print("remaining:", len(todo))

records = scores.to_dict("records")
SAVE_EVERY = 25

for i, row in enumerate(
    tqdm(
        todo.itertuples(index=False),
        total=len(todo),
        desc="WaveDetect",
    ),
    start=1,
):
    try:
        result = predictor.predict(row.text)
        score = float(result["ai_score"])
        error = None
    except Exception as exc:
        score = np.nan
        error = f"{type(exc).__name__}: {exc}"

    records.append({
        "row_id": row.row_id,
        "wave_ai_score": score,
        "wave_error": error,
    })

    if i % SAVE_EVERY == 0:
        (
            pd.DataFrame(records)
            .drop_duplicates("row_id", keep="last")
            .to_parquet(OUTPUT_PATH, index=False)
        )

scores = (
    pd.DataFrame(records)
    .drop_duplicates("row_id", keep="last")
)

scores.to_parquet(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
print("shape:", scores.shape)
print("successful:", scores["wave_ai_score"].notna().sum())
print("failed:", scores["wave_ai_score"].isna().sum())

already scored: 0
remaining: 9000


WaveDetect: 100%|██████████████████████████████████████████| 9000/9000 [06:29<00:00, 23.08it/s]

saved: ../results/wavedetect_scores.parquet
shape: (9000, 3)
successful: 9000
failed: 0


## Score audit

The paper's complementarity experiment uses the WaveDetect probability only after converting it to a clipped logit:

\[
W=\log\frac{p}{1-p}.
\]

That transformation belongs in the downstream detector notebook, not here. This file remains the raw public-model score cache.

In [7]:
audit = (
    pool[["row_id", "source", "domain", "generator", "label"]]
    .merge(scores, on="row_id", validate="one_to_one")
)

display(
    audit.groupby(["source", "domain", "generator"])["wave_ai_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

if audit["wave_error"].notna().any():
    display(
        audit[audit["wave_error"].notna()][
            ["row_id", "wave_error"]
        ].head(20)
    )

,source,domain,generator,count,mean,std,min,max
0,claude_eval,essay,claude,1000,0.998753,0.010478,0.817563,1.000000
1,claude_eval,reuter,claude,1000,0.996927,0.014908,0.716901,1.000000
2,claude_eval,wp,claude,1000,0.974595,0.094583,0.050866,1.000000
3,original,essay,gpt,1000,0.999964,0.000446,0.987779,1.000000
4,original,essay,human,1000,0.575060,0.379479,0.008427,1.000000
5,original,reuter,gpt,1000,0.995450,0.052535,0.138649,1.000000
6,original,reuter,human,1000,0.239218,0.329583,0.003599,0.999997
7,original,wp,gpt,1000,0.974314,0.111877,0.120028,1.000000
8,original,wp,human,1000,0.373098,0.376533,0.004195,1.000000


## Next step

After separately recomputing \(C\) and \(F\) from the same raw `ghostbuster_detector_pool.parquet`, merge the three outputs by `row_id` and run `05_wavedetect_complementarity.ipynb`.

This gives the auditable chain:

```text
public Ghostbuster text
    ├── Qwen scorer          -> C
    ├── recurrence extractor -> F
    └── public WaveDetect     -> W
                                  ↓
                       20-split complementarity
```